<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/08-operations/01-observability-and-llmops.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 可观测性与 LLMOps

**目标：** 给一个 LLM 应用装上探针，让你能看见它在生产里实际在做什么：追踪每一次调用、安全地记录它、跟踪成本/延迟/错误，并在用户报障之前就发现质量漂移。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 准备工作

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并拉入 `groq` 客户端——这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）；在 Colab 里通过**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，并打开 notebook access。在本地则把 `GROQ_API_KEY` 设为环境变量。

（完整讲解见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 加载 GROQ_API_KEY 并返回一个就绪的 Groq 客户端。
client, MODEL = setup()

## 为什么可观测性是本职工作，而不是附加项

评测（第 04 节）在交付*之前*告诉你系统是好的。可观测性在交付*之后*告诉你它*仍然是*好的，面对的是你从未预想过的真实流量。LLM 应用的失败方式和普通服务不同：当模型悄悄开始给出更差的答案、跑题，或一次 prompt 改动拖垮了某一类用户的质量时，不会有异常抛出。你唯一能发现的方式，就是**给它装上探针让它告诉你。**

核心单元是 **trace**：一次交互的结构化记录——输入、输出、token 用量、延迟，以及足够的 metadata，以便日后切片分析。LLMOps 里的一切都建立在拥有好的 trace 之上：成本看板、延迟 SLO、漂移检测，以及回到评测集的那条反馈回路。我们会从零搭一个最小 tracer。能迁移的是概念；托管工具只是把它做得好看些。

## 一个最小 tracer

不需要框架。一条 trace 就是每次调用的一个 dict，装着日后你会想查询的那些字段。纪律是：*每一次*调用都捕获它，无论成功还是失败，并且绝不让追踪本身抛异常。

In [ ]:
import time, json, uuid

TRACES = []   # 在真实应用里这是一个 DB / 日志管线，而不是一个 list

def traced_call(messages, tag=None, **kwargs):
    """包装一次 Groq 调用，并记录一条结构化 trace。追踪本身绝不抛异常。"""
    trace = {"id": str(uuid.uuid4())[:8], "tag": tag, "model": kwargs.get("model", MODEL)}
    t0 = time.monotonic()
    try:
        resp = client.chat.completions.create(model=kwargs.pop("model", MODEL),
                                               messages=messages, **kwargs)
        trace["latency_s"] = round(time.monotonic() - t0, 3)
        trace["ok"] = True
        trace["prompt_tokens"] = resp.usage.prompt_tokens
        trace["completion_tokens"] = resp.usage.completion_tokens
        trace["output"] = resp.choices[0].message.content
        return resp
    except Exception as e:
        trace["latency_s"] = round(time.monotonic() - t0, 3)
        trace["ok"] = False
        trace["error"] = f"{type(e).__name__}: {e}"
        raise
    finally:
        TRACES.append(trace)   # 无论这次调用成功还是炸了，都捕获下来

# 制造一些流量
for q in ["What is TCP?", "Explain DNS in one sentence.", "What port does HTTPS use?"]:
    traced_call([{"role": "user", "content": q}], tag="demo", max_tokens=80)

print(json.dumps(TRACES[-1], indent=2)[:400])

## 安全地记录 prompt：日志是一条泄漏路径（接安全那节）

那个显然的做法——把完整 prompt 和输出都写日志——也是一个**随时可能发生的 PII/密钥泄漏**（OWASP LLM02，第 07 节）。prompt 往往含有用户数据；输出有时会把它们回显出来。粗心的日志会安静地把这些送到你日志去的任何地方。两条规则：

- **记录之前先脱敏。** 掩掉邮箱、电话号、像卡号的连续数字。
- **截断并采样。** 你很少需要每一次调用的完整文本；存一个预览 + hash，完整负载只保留给采样到的一小部分或出错的情况。

In [ ]:
import re, hashlib

def redact(text):
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', '<email>', text)
    text = re.sub(r'\b(?:\d[ -]?){13,16}\b', '<card>', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '<phone>', text)
    return text

def safe_log(text, keep=120):
    red = redact(text)
    return {"preview": red[:keep], "chars": len(text),
            "sha1": hashlib.sha1(text.encode()).hexdigest()[:10]}  # 用 hash 去重/追踪，而不存原文

sample = "Contact me at jane.doe@acme.com or 555-123-4567 about card 4111 1111 1111 1111."
print(safe_log(sample))

现在日志行带着一段脱敏后的预览、一个长度、一个可以用来把相同 prompt 分组的 hash——足够调试和分析，又不会把你的日志存储变成 PII 泄漏源。（当你确实需要完整负载时，它们要放在访问控制和保留期窗口之后。）

## 真正要紧的指标

从一堆 trace 里，四个数字告诉你系统是否健康。持续计算它们，并对*变化量*告警，而不是对绝对值：

- **成本。** token 数 × 单价，聚合起来（`01-model-apis/04` 里的算法）。提防悄悄爬升：一次 prompt 调整加了 2k token × 数百万次调用，那是一个预算级事件。
- **延迟。** 跟踪 p50 *和* p95/p99。平均值会掩盖真正让用户烦躁的尾部；对流式 UI（`01-model-apis/03`），TTFT 很重要。
- **错误/失败率。** API 错误、超时，*以及* schema 校验失败（一次「成功」但返回不可解析 JSON 的调用，对你的应用来说就是失败）。
- **质量代理指标。** 你没法在真实流量上跑完整评测，但有廉价的信号能跟踪漂移：拒答率、输出长度分布、通过校验的工具调用占比、点赞/点踩（如果你收集的话）。

In [ ]:
import statistics

def report(traces):
    ok = [t for t in traces if t.get("ok")]
    n = len(traces)
    lat = sorted(t["latency_s"] for t in ok)
    def pct(p):
        return lat[min(len(lat)-1, int(len(lat)*p))] if lat else 0
    tokens = sum(t.get("prompt_tokens",0)+t.get("completion_tokens",0) for t in ok)
    print(f"calls            : {n}")
    print(f"error rate       : {(n-len(ok))/n:.0%}" if n else "n/a")
    print(f"latency p50 / p95: {pct(0.5):.3f}s / {pct(0.95):.3f}s")
    print(f"total tokens     : {tokens:,}")
    # 示例性成本：$0.90 / 1M token（真实算法见 01-model-apis/04）
    print(f"est. cost        : ${tokens * 0.90 / 1_000_000:.5f}")

report(TRACES)

## 闭上回路：可观测性 → 评测

追踪的回报不是那块看板，而是那条**反馈回路**。生产会暴露出你从没想象过的输入，而每一个坏输入都是一个新评测用例：

1. 某条 trace 看起来不对劲（质量代理指标很低、用户点踩、一则支持投诉）。
2. 你从 trace 里把那个确切的输入捞出来。
3. 它成为你黄金集（`04-evals/01`）里的一个 case，带上它*本该*给出的答案。
4. 你的回归 harness（`04-evals/03`）从此永远守住它。

这就是整个 LLMOps 飞轮：**交付前评测 → 在生产中观测 → 把失败喂回评测。** 每一个 bug 都变成一个测试，和后端回归测试是同一套纪律，只是作用在模型行为上。

**工具在哪里介入：** 托管平台（**LangSmith、Langfuse、Arize Phoenix、Braintrust、Helicone**）开箱就给你这套 tracer、脱敏、看板，以及 trace 到评测的接线，还有跨周的趋势历史。当 `TRACES = []` 这个 list 不再能扩展时（需要看数据的不止一个人了），就采纳一个。正因为你在这里亲手搭了最小版本，你才会评估得好。

## 实践与反模式

| ✅ 该做 | ❌ 反模式 |
|---|---|
| 追踪每一次调用（入/出/token/延迟/状态），无论成功*还是*失败 | 什么都不埋点，等客户投诉才知道 |
| 记录前先脱敏；对完整负载采样 | 把原始 prompt/输出写日志：一次安静的 PII/密钥泄漏 |
| 跟踪 p50 **和** p95/p99；对变化量告警 | 只看均值；错过用户真正感受到的尾部 |
| 跟踪质量代理指标（拒答率、长度、校验通过率）来发现漂移 | 以为「没有异常」就等于质量没问题 |
| 把生产失败喂回黄金集（那个飞轮） | 把看板当作终点；从不闭合回路 |
| 当 trace list 不再能扩展时采纳托管平台 | 在还不知道它能自动化什么之前就上 LangSmith/Langfuse |

（全仓库范围的清单见 [docs/best-practices-and-anti-patterns.md](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/docs/best-practices-and-anti-patterns.md)。）

## 练习

1. **打标签并切片。** 给每条 trace 加一个 `user_tier` 字段（"free"/"paid"），并按 tier 计算延迟 p95 和错误率。各分群之间不均衡的痛苦在全局均值里是看不见的，这就是为什么要切片。
2. **对变化量告警。** 写一个 `check_drift(window_a, window_b)`，当平均输出长度或拒答率在两批 trace 之间变化超过 X% 时报警。这就是十来行代码的漂移检测。
3. **端到端追踪一次 RAG 调用。** 给 `03-rag` 里的 `answer()` 函数装上探针，让一条 trace 同时捕获 query、检索到的 doc id，*以及*生成结果。这样当某个答案很差时，你能直接从日志里判断这是检索的锅还是生成的锅（`03-rag/04-why-rag-fails` 里的那个区分）。
4. **采样策略。** 修改 `traced_call`，对 100% 的错误存完整负载，但对成功只存 10%。全都存下来的成本 vs 什么都不存的调试能力：你的界线画在哪，为什么？